# Module 5 lab · Many variables and gradients

The code from every lesson of [this module](https://aicourses.dev/mathematics/gradients), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 5.2 Partial derivatives: one knob at a time

Hold every knob still but one and you are back to an ordinary slope. This lesson takes both partial slopes of the penguin line’s error.

[Read the lesson](https://aicourses.dev/mathematics/gradients/partial-derivatives)

### In NumPy

Load the twelve birds, centred as in lesson 04.8, and write the error as a function of both knobs. Then check the three places used above.

In [ ]:
import numpy as np

flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)     # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
x = (flipper - flipper.mean()) / 10    # cm above the average
y = (mass - mass.mean()) / 1000        # kg above the average
L = lambda w, b: np.mean((w * x + b - y) ** 2)
print(round(L(0, 0), 4), round(L(0, 1), 4), round(L(0.4891, 0), 4))

*The lesson shows:*

```text
0.5454 1.5454 0.1828
```

The central difference of lesson 04.9 gives a partial slope when you let one knob move and pass the other as a fixed number:

In [ ]:
def central(f, t, h=1e-5):
    return (f(t + h) - f(t - h)) / (2 * h)

for w, b in [(0, 1), (0.5, 0.5), (1, -0.5)]:
    dw = central(lambda t: L(t, b), w)    # w moves, b held still
    db = central(lambda t: L(w, t), b)    # b moves, w held still
    print(w, b, round(dw, 4), round(db, 4))

*The lesson shows:*

```text
0 1 -1.4823 2.0
0.5 0.5 0.0329 1.0
1 -0.5 1.5481 -1.0
```

The second row is the “your turn” answer. Now the formulas, first as a loop over the birds and then in one line each:

In [ ]:
def partials(w, b):
    dw = db = 0.0
    for xi, yi in zip(x, y):              # one bird at a time
        miss = w * xi + b - yi
        dw += 2 * miss * xi / 12
        db += 2 * miss / 12
    return round(dw, 4), round(db, 4)

miss = lambda w, b: w * x + b - y
print(*partials(0, 1), *partials(0.5, 0.5))
print(round(2 * np.mean(miss(0, 1) * x), 4), round(2 * np.mean(miss(0, 1)), 4))

*The lesson shows:*

```text
-1.4823 2.0 0.0329 1.0
-1.4823 2.0
```

The loop adds each bird’s slope divided by 12, which is the mean; the one-liners use NumPy’s `mean`. Both agree with the central differences. Finally, check the gift of centring: leave the tilt at 0 and move the shift.

In [ ]:
for b in [0, 0.5, 1]:
    print(b, round(central(lambda t: L(t, b), 0), 4))

*The lesson shows:*

```text
0 -1.4823
0.5 -1.4823
1 -1.4823
```

## 5.3 The gradient

The partial slopes, gathered into one vector, are the gradient: the direction of steepest climb and how steep it is. Gradient descent in Module 06 is built on it.

[Read the lesson](https://aicourses.dev/mathematics/gradients/the-gradient)

### One entry per knob

The gradient has as many entries as there are knobs. For the line, two; for the network of this module’s project, 1,210, one slope for each of its weights and shifts. Nothing else changes: the same arrow idea holds in 1,210 directions, even though no one can draw it. The code below builds the penguin gradient from the formulas of lesson 2, and checks it by nudging each knob in turn:

In [ ]:
import numpy as np

flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)     # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
x = (flipper - flipper.mean()) / 10    # cm above the average
y = (mass - mass.mean()) / 1000        # kg above the average
L = lambda w, b: np.mean((w * x + b - y) ** 2)

def grad(w, b):
    miss = w * x + b - y
    return np.array([2 * np.mean(miss * x), 2 * np.mean(miss)])

g = grad(0, 1)
print(np.round(g, 4), round(np.linalg.norm(g), 4))

*The lesson shows:*

```text
[-1.4823  2.    ] 2.4894
```

`np.linalg.norm` is the length. Now the check, which works for any number of knobs. It is a loop over the knobs, and each one needs two evaluations of the error:

In [ ]:
def num_grad(f, p, h=1e-5):
    g = np.zeros(len(p))
    for i in range(len(p)):               # one knob at a time
        e = np.zeros(len(p)); e[i] = h
        g[i] = (f(*(p + e)) - f(*(p - e))) / (2 * h)
    return g

print(np.round(num_grad(L, np.array([0.0, 1.0])), 4))
best = np.sum(x * y) / np.sum(x * x)
print(np.round(grad(0.5, 0.5), 4), np.round(abs(grad(best, 0)), 4))

*The lesson shows:*

```text
[-1.4823  2.    ]
[0.0329 1.    ] [0. 0.]
```

## 5.4 Which way is downhill?

Why is downhill the opposite of the gradient, and why does the gradient cross contour lines at right angles? This lesson answers with the dot product from Module 02.

[Read the lesson](https://aicourses.dev/mathematics/gradients/which-way-is-downhill)

### In NumPy

Start by building the penguin error and its gradient from lesson 3, and print the gradient at the first point:

In [ ]:
import numpy as np

flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222], dtype=float)     # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
x = (flipper - flipper.mean()) / 10    # cm above the average
y = (mass - mass.mean()) / 1000        # kg above the average
L = lambda w, b: np.mean((w * x + b - y) ** 2)

def grad(w, b):
    miss = w * x + b - y
    return np.array([2 * np.mean(miss * x), 2 * np.mean(miss)])

print(np.round(grad(0, 1), 4))

*The lesson shows:*

```text
[-1.4823  2.    ]
```

The slope in each of four directions from $(0, 1)$, by the dot product and by a central difference along the path; the last direction is the one at a right angle:

In [ ]:
def central(f, t, h=1e-5):
    return (f(t + h) - f(t - h)) / (2 * h)

g = grad(0, 1)
n = np.linalg.norm(g)
dirs = {"along w": (1, 0), "(0.6, 0.8)": (0.6, 0.8),
        "gradient": g / n, "across": (g[1] / n, -g[0] / n)}
for name, u in dirs.items():
    path = lambda t, u=u: L(0 + t * u[0], 1 + t * u[1])
    print(name, round(central(path, 0), 4) + 0.0, round(g @ u, 4) + 0.0)

*The lesson shows:*

```text
along w -1.4823 -1.4823
(0.6, 0.8) 0.7106 0.7106
gradient 2.4894 2.4894
across 0.0 0.0
```

(The `+ 0.0` turns a rounded `-0.0` into `0.0`.) The two columns agree in every direction: the slope along the gradient is its length, and across it the slope is 0. Now one small step against the gradient, with the first-order prediction, then the direction to the bottom for comparison:

In [ ]:
eta = 0.01
p = np.array([0.0, 1.0])
step = p - eta * g
print(round(L(*p), 4), round(L(*step), 4), round(L(*p) - eta * g @ g, 4))
to_bottom = np.array([0.4891, 0]) - p
print(np.round(-g / n, 4), np.round(to_bottom / np.linalg.norm(to_bottom), 4))

*The lesson shows:*

```text
1.5454 1.4841 1.4834
[ 0.5954 -0.8034] [ 0.4394 -0.8983]
```

The step lowers the error from 1.5454 to 1.4841, and the first-order prediction says 1.4834; the small gap is the curvature, which the straight-line guess leaves out. The two arrows on the second line are not the same. Finally the “your turn” answer:

In [ ]:
q = grad(0.5, 0.5)
u = np.array([0.8, -0.6])
path = lambda t: L(0.5 + t * u[0], 0.5 + t * u[1])
print(round(q @ u, 4), round(central(path, 0), 4))

*The lesson shows:*

```text
-0.5737 -0.5737
```

## 5.5 The chain rule when paths split

When a knob reaches the output by two routes, the slopes along the routes add up. This extends the chain rule from Module 04 to the shapes real networks have.

[Read the lesson](https://aicourses.dev/mathematics/gradients/chain-rule-many-paths)

### When a value feeds many things

The route through $o_1$ has slopes $2(o_1 - 1) = 6$, then $2$, then $x = 2$: $6 \times 2 \times 2 = 24$. The route through $o_2$ has $2 o_2 = -12$, then $-3$, then $x = 2$: $-12 \times (-3) \times 2 = 72$. The total is 96. In the first-layer weights of this module’s project network each weight reaches the loss by 10 routes, one through each output, and its slope is the sum of ten such products.

In [ ]:
import numpy as np

def central(f, t, h=1e-5):
    return (f(t + h) - f(t - h)) / (2 * h)

p = lambda w: 3 * w + 1
q = lambda w: w ** 2
f = lambda w: p(w) * q(w)
w = 2
via_p = q(w) * 3          # p moves 3 per unit of w, L feels it q times
via_q = p(w) * 2 * w      # q moves 2w per unit of w, L feels it p times
print(via_p, via_q, via_p + via_q, round(central(f, w), 4))

*The lesson shows:*

```text
12 28 40 40.0
```

The same for the value that feeds two outputs:

In [ ]:
x = 2
h = lambda w: w * x
o1 = lambda w: 2 * h(w)
o2 = lambda w: -3 * h(w)
loss = lambda w: (o1(w) - 1) ** 2 + o2(w) ** 2
w = 1
route1 = 2 * (o1(w) - 1) * 2 * x      # through o1
route2 = 2 * o2(w) * (-3) * x         # through o2
print(loss(w), route1, route2, route1 + route2, round(central(loss, w), 4))

*The lesson shows:*

```text
45 24 72 96 96.0
```

## 5.6 Computation graphs

Any calculation is small steps wired together, and each step knows its own slope. Drawing it as a graph is what makes backpropagation mechanical.

[Read the lesson](https://aicourses.dev/mathematics/gradients/computation-graphs)

### The slow way to the slopes

What the graph does not yet give you is the slope of $L$ with respect to $p$ and $q$. You do know one way to get it: nudge each input and run the whole forward pass again, as in lesson 04.9. Here is the graph in code, in three blocks. First the forward pass, written as a list of steps (the code calls the last one `v5`; it is $L$):

In [ ]:
p, q = 2.0, 3.0
v1 = p * q
v2 = p + q
v3 = v1 * v2
v4 = v3 - 25
v5 = v4 ** 2              # this is L
print(v1, v2, v3, v4, v5)

*The lesson shows:*

```text
6.0 5.0 30.0 5.0 25.0
```

Then the local slopes, each from the values above:

In [ ]:
d1 = (q, p)          # v1 = p * q: slope against p, against q
d2 = (1, 1)          # v2 = p + q
d3 = (v2, v1)        # v3 = v1 * v2: slope against v1, against v2
d4 = 1               # v4 = v3 - 25
d5 = 2 * v4          # v5 = v4 ** 2
print(d1, d2, d3, d4, d5)

*The lesson shows:*

```text
(3.0, 2.0) (1, 1) (5.0, 6.0) 1 10.0
```

Then the slow way: run the whole calculation again for each nudge, and count the runs.

In [ ]:
passes = 0
def run(p, q):
    global passes
    passes += 1
    return ((p * q) * (p + q) - 25) ** 2

h = 1e-5
dp = (run(p + h, q) - run(p - h, q)) / (2 * h)
dq = (run(p, q + h) - run(p, q - h)) / (2 * h)
print(round(dp, 3), round(dq, 3), passes)

*The lesson shows:*

```text
210.0 160.0 4
```

## 5.7 Backpropagation

Running the chain rule backwards once gives every knob’s slope for about the cost of one forward pass. This is the reason deep learning is affordable.

[Read the lesson](https://aicourses.dev/mathematics/gradients/backpropagation)

### In NumPy

The backward sweep for the graph at $p = 2$, $q = 3$, line by line, with `s` for the signal at a dot:

In [ ]:
import numpy as np

p, q = 2.0, 3.0
v1, v2 = p * q, p + q
v3 = v1 * v2
v4 = v3 - 25
v5 = v4 ** 2
s5 = 1.0                      # signal at the end: L against L
s4 = s5 * 2 * v4              # local slope of the square
s3 = s4 * 1
s1, s2 = s3 * v2, s3 * v1     # the two factors of v3
print(s5, s4, s3, s1, s2)

*The lesson shows:*

```text
1.0 10.0 10.0 50.0 60.0
```

In [ ]:
sp = s1 * q + s2 * 1          # p reaches L by two routes: add them
sq = s1 * p + s2 * 1
print(sp, sq)

*The lesson shows:*

```text
210.0 160.0
```

They equal the central differences of lesson 6. Now the penguin line again: one bird, two knobs, a graph of four steps, $a_1 = w x$, $a_2 = a_1 + b$, $a_3 = a_2 - y$ and the loss $a_4 = a_3^2$. The twelfth bird at $w = 0$, $b = 1$ is the one you met in lesson 2, whose slopes were $-2.3644$ and $-0.975$:

In [ ]:
x12, y12 = 2.425, 1.4875      # the twelfth bird
w, b = 0.0, 1.0
a1 = w * x12
a2 = a1 + b
a3 = a2 - y12
a4 = a3 ** 2
t4 = 1.0
t3 = t4 * 2 * a3              # signal at the miss
t2 = t3                       # subtracting y passes it on
t1 = t2                       # so does adding b
print(round(a4, 4), round(t2, 4), round(t1 * x12, 4))

*The lesson shows:*

```text
0.2377 -0.975 -2.3644
```

## 5.8 Jacobians: keeping track of shapes

When a machine has many inputs and many outputs, its slopes form a table. The lesson shows how to keep track of shapes so the gradient you carry back always fits the thing it belongs to.

[Read the lesson](https://aicourses.dev/mathematics/gradients/jacobians)

### In NumPy

Build the table of the matrix machine by nudging one input at a time, then carry the signal back and check it with a loss that has those slopes:

In [ ]:
import numpy as np

W = np.array([[1, 2], [0, 1], [-1, 1]], dtype=float)   # 3 rows, 2 columns
x = np.array([2.0, 1.0])
z = lambda x: W @ x
J = np.zeros((3, 2))
for j in range(2):                     # nudge one input at a time
    e = np.zeros(2); e[j] = 1e-5
    J[:, j] = (z(x + e) - z(x - e)) / 2e-5
print(np.round(J, 4), J.shape)

*The lesson shows:*

```text
[[ 1.  2.]
 [ 0.  1.]
 [-1.  1.]] (3, 2)
```

The nudged table is the matrix. Now the signal:

In [ ]:
signal = np.array([1.0, 0.0, -2.0])    # slope of the loss against each output
print(W.T @ signal)                    # carried back to the input
loss = lambda x: signal @ z(x)         # a loss with exactly these slopes
for j in range(2):
    e = np.zeros(2); e[j] = 1e-5
    print(round((loss(x + e) - loss(x - e)) / 2e-5, 4) + 0.0)

*The lesson shows:*

```text
[3. 0.]
3.0
0.0
```

For the tanh, the table of slopes is diagonal, and the diagonal is the curve’s slope:

In [ ]:
zz = np.array([0.5, 0.0, -1.0])
J = np.zeros((3, 3))
for j in range(3):
    e = np.zeros(3); e[j] = 1e-5
    J[:, j] = (np.tanh(zz + e) - np.tanh(zz - e)) / 2e-5
print(np.round(J, 4))
print(np.round(1 - np.tanh(zz) ** 2, 4))

*The lesson shows:*

```text
[[0.7864 0.     0.    ]
 [0.     1.     0.    ]
 [0.     0.     0.42  ]]
[0.7864 1.     0.42  ]
```

## 5.9 The gradients of a layer

You find the gradients of a layer, a matrix, a shift and a loss, and check each entry with numbers. They are the building block of the project that follows.

[Read the lesson](https://aicourses.dev/mathematics/gradients/gradients-of-a-layer)

### Entry by entry

Here is the same in NumPy. First the outputs and the three gradients, with `np.outer` for the outer product:

In [ ]:
import numpy as np

W = np.array([[1, 2, 0], [0, 1, -1]], dtype=float)    # 2 outputs, 3 inputs
b = np.array([0.5, -1.0])
x = np.array([1.0, 2.0, 3.0])
t = np.array([5.0, -1.0])
z = W @ x + b
s = z - t                                             # signal at the output
print(z, s, 0.5 * np.sum(s ** 2))
print(np.outer(s, x))                                 # slopes of W
print(s, W.T @ s)                                     # slopes of b and of x

*The lesson shows:*

```text
[ 5.5 -2. ] [ 0.5 -1. ] 0.625
[[ 0.5  1.   1.5]
 [-1.  -2.  -3. ]]
[ 0.5 -1. ] [0.5 0.  1. ]
```

Now check every entry of all three with central differences, nudging one entry at a time:

In [ ]:
loss = lambda: 0.5 * np.sum((W @ x + b - t) ** 2)

def numeric(a, h=1e-5):                  # slope for every entry of a
    g = np.zeros_like(a)
    for i in np.ndindex(a.shape):
        old = a[i]
        a[i] = old + h; up = loss()
        a[i] = old - h; down = loss()
        a[i] = old
        g[i] = (up - down) / (2 * h)
    return g

print(np.round(numeric(W), 4) + 0.0)
print(np.round(numeric(b), 4), np.round(numeric(x), 4) + 0.0)

*The lesson shows:*

```text
[[ 0.5  1.   1.5]
 [-1.  -2.  -3. ]]
[ 0.5 -1. ] [0.5 0.  1. ]
```

### A batch at once

Check the shapes in each line: $(2, B) \times (B, 3)$ gives $(2, 3)$, and $(B, 2) \times (2, 3)$ gives $(B, 3)$. For $B = 1$, `S.T @ X` is the outer product $\mathbf{s}\,\mathbf{x}^\top$ above. Here is a random batch of 4, with every gradient checked against `numeric`:

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(4, 3))               # a batch of 4 inputs, as rows
T = rng.normal(size=(4, 2))
Z = X @ W.T + b                           # shape (4, 2)
S = (Z - T) / len(X)                      # mean loss: divide by batch size
dW, db, dX = S.T @ X, S.sum(axis=0), S @ W
print(dW.shape, db.shape, dX.shape)

*The lesson shows:*

```text
(2, 3) (2,) (4, 3)
```

In [ ]:
loss = lambda: 0.5 * np.sum((X @ W.T + b - T) ** 2) / len(X)
gaps = [np.abs(dW - numeric(W)).max(), np.abs(db - numeric(b)).max(),
        np.abs(dX - numeric(X)).max()]
print(all(g < 1e-8 for g in gaps))

*The lesson shows:*

```text
True
```

## 5.10 Project: backpropagation from scratch

You write backpropagation for a two-layer network in NumPy and check every gradient against a finite difference. When it passes, you have built the engine every framework hides.

[Read the lesson](https://aicourses.dev/mathematics/gradients/project-backprop-from-scratch)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: set up the problem

Build the digits split of Module 02 and the tiny problem: four points whose targets are 0, 1, 1, 0, the exclusive-or of lesson 03.8, which one layer cannot solve and a hidden layer with a bend can. Start the knobs small and random, and the shifts at zero.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                               # pixels 0 to 1
order = rng.permutation(len(X))
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]            # the 1,297 to learn from

P = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])   # four made-up points
T = np.array([[0.], [1.], [1.], [0.]])                    # what each should give
W1 = rng.normal(0, 0.5, size=(3, 2)); b1 = np.zeros(3)   # 2 in, 3 hidden
W2 = rng.normal(0, 0.5, size=(1, 3)); b2 = np.zeros(1)   # 3 hidden, 1 out
tiny = [W1, b1, W2, b2]
print(P.shape, T.shape, [a.shape for a in tiny])

*The lesson shows:*

```text
(4, 2) (4, 1) [(3, 2), (3,), (1, 3), (1,)]
```

### Step 2: the forward pass and the loss

Write `forward` for the layers in the form of lesson 03.8, `X @ W.T + b`, and `loss` for half the squared miss added over the outputs and averaged over the examples.

In [ ]:
def forward(p, X):
    W1, b1, W2, b2 = p
    H = np.tanh(X @ W1.T + b1)          # hidden layer, with the bend
    return H, H @ W2.T + b2             # and the output layer

def loss(p, X, T):                      # half the squared misses, averaged
    return 0.5 * np.mean(np.sum((forward(p, X)[1] - T) ** 2, axis=1))

H, out = forward(tiny, P)
print(np.round(out[:, 0], 4), round(loss(tiny, P, T), 6))

*The lesson shows:*

```text
[ 0.      0.0873 -0.2297 -0.1531] 0.296079
```

### Step 3: the backward pass

Work from the end, as in lessons 7 and 9. The signal at the output is the miss divided by the number of examples. Pass it through the second layer with `S @ W2` and through the tanh by multiplying by its slope $1 - H^2$ (lesson 04.6), then read the four gradients off with the formulas of lesson 9. Check each shape as you go: `s_out` is (4, 1) and `W2` is (1, 3), so `s_out @ W2` is (4, 3), the shape of `H`; and `s_hid.T @ X` is (3, 4) times (4, 2), which is (3, 2), the shape of `W1`.

In [ ]:
def backward(p, X, T):
    W1, b1, W2, b2 = p
    H, out = forward(p, X)
    s_out = (out - T) / len(X)              # signal at the output
    s_hid = (s_out @ W2) * (1 - H ** 2)     # back through W2, then the tanh
    return [s_hid.T @ X, s_hid.sum(axis=0), s_out.T @ H, s_out.sum(axis=0)]

for name, g in zip(["W1", "b1", "W2", "b2"], backward(tiny, P, T)):
    print(name, np.round(g, 5).tolist())
print(round(np.mean(out - T), 5))           # b2's gradient, by hand

*The lesson shows:*

```text
W1 [[-0.00938, -0.00778], [-0.28111, -0.24666], [-0.06689, -0.05894]]
b1 [-0.01623, -0.49153, -0.1143]
W2 [[-0.14891, 0.08424, -0.07244]]
b2 [-0.57388]
-0.57388
```

### Step 4: check every gradient

Nudge each knob in turn, up and down by $h$, and compare the central difference of lesson 04.9 with the backward pass. Put each entry back after nudging it.

In [ ]:
def numeric(p, X, T, h=1e-5):
    out = []
    for a in p:
        g = np.zeros_like(a)
        for i in np.ndindex(a.shape):
            old = a[i]
            a[i] = old + h; up = loss(p, X, T)
            a[i] = old - h; down = loss(p, X, T)
            a[i] = old
            g[i] = (up - down) / (2 * h)
        out.append(g)
    return out

def worst(p, X, T, h=1e-5):             # largest gap, backprop against nudge
    return max(np.abs(a - b).max()
               for a, b in zip(backward(p, X, T), numeric(p, X, T, h)))
for h in [1e-1, 1e-3, 1e-5, 1e-8, 1e-10]:
    print(h, "%.2e" % worst(tiny, P, T, h))

*The lesson shows:*

```text
0.1 1.61e-03
0.001 1.62e-07
1e-05 1.47e-11
1e-08 4.36e-09
1e-10 3.70e-07
```

The gap falls like $h^2$ down to $h = 10^{-5}$, where it is 1.5e-11, and then rises as rounding takes over, the table of lesson 04.9 again. Now see that the check can fail: slip 1% into one entry of the gradient.

In [ ]:
bad = backward(tiny, P, T)
bad[0][1, 1] *= 1.01                    # a 1% slip in one entry of W1's gradient
print("%.2e" % max(np.abs(a - b).max() for a, b in zip(bad, numeric(tiny, P, T))))

*The lesson shows:*

```text
2.47e-03
```

### Step 5: the digits

The same three functions work for 64-16-10, since nothing in them knows the sizes. Make the knobs, use the one-hot targets (row $k$ of the identity matrix for the digit $k$) and check one training image, which has 1,210 knobs to check:

In [ ]:
V1 = rng.normal(0, 0.1, size=(16, 64)); c1 = np.zeros(16)
V2 = rng.normal(0, 0.1, size=(10, 16)); c2 = np.zeros(10)
big = [V1, c1, V2, c2]
Y = np.eye(10)                          # one-hot targets
x1, t1 = Xtr[:1], Y[ytr[:1]]            # one training image
print(ytr[0], sum(a.size for a in big), round(loss(big, x1, t1), 6))
print("%.2e" % worst(big, x1, t1))

*The lesson shows:*

```text
6 1210 0.554026
1.55e-11
```

The image is a 6, the network has 1,210 knobs, and the largest of its 1,210 gaps is 1.55e-11. The check took $2 \times 1{,}210 = 2{,}420$ forward passes; the backward pass took one forward and one backward. Now the same check in float32, the precision of lesson 01.9:

In [ ]:
big32 = [a.astype(np.float32) for a in big]
g32 = numeric(big32, x1.astype(np.float32), t1.astype(np.float32))
gaps = [np.abs(a - b).max() for a, b in zip(backward(big, x1, t1), g32)]
print("%.2e" % max(gaps))

*The lesson shows:*

```text
8.11e-03
```

Rounding to 7 digits swamps a nudge of $10^{-5}$: the gap is 8.1e-3, as much as a wrong gradient would give. Do gradient checks in float64. Last, one plain step of $0.1$ against the gradient, on all 1,297 training images, to see the loss fall (Module 06 makes it a loop):

In [ ]:
Xb, Tb = Xtr, Y[ytr]                    # all 1,297 training images
g = backward(big, Xb, Tb)
new = [a - 0.1 * ga for a, ga in zip(big, g)]
print(round(loss(big, Xb, Tb), 4), round(loss(new, Xb, Tb), 4))

*The lesson shows:*

```text
0.6362 0.5392
```